# F5-M00: Índice — Modelado Clásico

**TFM: Predicción de Abandono Universitario**

| | |
|---|---|
| **Autora** | María José Morte Ruiz |
| **Email** | mjmorteruiz@uoc.edu (UOC) / morte@uji.es (UJI) |

---

## Qué hace
Página de entrada de la Fase 5 (Modelado Clásico). Lee la configuración central
de `src.config_modelado`, carga el dataset de producción y genera el HTML de
índice con tarjetas dinámicas a los 5 módulos.

**Ningún valor está hardcodeado**: algoritmos, métricas, folds, random_state
y módulos se leen todos de `src.config_modelado`.

## Requisitos
- `data/03_features/dataset_final_tfm.parquet` — dataset de producción (origen: f3_m05)
- `src/config_modelado.py` — configuración central de Fase 5
- `src/config.py` — rutas, utilidades, colores del proyecto
- `src/display.py` — banners y visualizaciones animadas
- Entorno: `tfm_abandono`

## Genera
- `docs/html/fase5/fase5_index.html`

## Flujo
```
src/config_modelado.py  ←─ configuración central (algoritmos, CV, métricas)
data/03_features/dataset_final_tfm.parquet  ←─ Fase 3 (f3_m05)
        ↓
f5_m00  →  índice + HTML (esta tarjeta)
f5_m01a → preparación
f5_m01b → lineales básicos
f5_m01c → lineales ext (Perceptron·SGD·SGDElasticNet)
f5_m01d → comparativa lineal
f5_m02  →  árboles
f5_m03  →  gradient boosting
f5_m04  →  otros algoritmos
f5_m05  →  comparativa final → top-3 para Fase 6
```

## Siguiente
`f5_m01a_preparacion.ipynb` — Split + pipeline preprocesamiento

In [ ]:
# ============================================================================
# CELDA 1: CONFIGURACIÓN
# ============================================================================
# Detecta ROOT, añade al sys.path e importa los módulos del proyecto.
#
# REGLA sys.path: insertar siempre ROOT (no ROOT/src).
#   Correcto:   sys.path.insert(0, str(ROOT))
#   Incorrecto: sys.path.insert(0, str(ROOT / 'src'))
#   Python resuelve 'from src.config import ...' desde ROOT.
# ============================================================================

import sys
import warnings
from pathlib import Path

warnings.filterwarnings('ignore')

# ── Detección de entorno y ROOT ──────────────────────────────────────────────
ROOT = Path.cwd()
for _ in range(6):
    if (ROOT / 'src').exists():
        break
    ROOT = ROOT.parent

sys.path.insert(0, str(ROOT))  # ROOT, no ROOT/src

# ── Librerías estándar ────────────────────────────────────────────────────────
import pandas as pd

# ── Configuración del proyecto ────────────────────────────────────────────────
from src.config import RUTA_HTML, DATASET_MODELADO, info_entorno
from src.utils import crear_directorios, formato_numero_es
from src.utils.graficos import COLORES

# ── Configuración central de Fase 5 ─────────────────────────────────────────
# Todo lo de modelado viene de aquí — cero hardcodes en el notebook
from src.config_modelado import (
    CV_CONFIG,              # folds, random_state, test_size
    METRICAS_EVALUAR,       # lista de métricas con metadatos
    BASELINE_AUTOML,        # referencia del screening AutoML
    MODULOS_FASE5,          # módulos con algoritmos y descripciones dinámicas
    TODOS_LOS_ALGORITMOS,   # lista plana de todos los algoritmos
    ESTRATEGIAS_BALANCE,    # estrategias de balance de clases
    resumen_config,         # función de diagnóstico
)

# ── Sistema HTML ──────────────────────────────────────────────────────────────
# render_pagina_desde_fichero es el ESTÁNDAR para todos los notebooks.
# Extrae título, fase y módulo del nombre del fichero automáticamente.
# Solo usar render_base_html en casos excepcionales (título no inferible).
from src.html.render import render_pagina_desde_fichero
from src.html import (
    generar_kpis_html,
    generar_tarjetas_html,
    generar_seccion_html,
    guardar_html,
)

# ── Visualizaciones animadas ───────────────────────────────────────────────────
from src.display import mostrar_banner, mostrar_resumen_final

# ── Rutas ──────────────────────────────────────────────────────────────────────
RUTA_FASE5_HTML = RUTA_HTML / 'fase5'
crear_directorios([RUTA_FASE5_HTML])

fmt = formato_numero_es  # Atajo: fmt(33621) → '33.621'

# ── Diagnóstico ───────────────────────────────────────────────────────────────
info_entorno()
resumen_config()

In [ ]:
# ============================================================================
# CELDA 2: CARGAR DATASET Y CALCULAR KPIs
# ============================================================================
# Carga el dataset de producción y extrae métricas para el HTML.
#
# dataset_final_tfm.parquet = D_strict (f3_m05), auditoría de leakage.
# 24 features + target binario 'abandono', sin variables contaminadas.
# ============================================================================

# ── Carga ────────────────────────────────────────────────────────────────────
df = pd.read_parquet(DATASET_MODELADO)

# ── Métricas del dataset (calculadas, no hardcodeadas) ────────────────────────
n_filas, n_cols   = df.shape
n_features        = n_cols - 1
pct_abandono      = df['abandono'].mean() * 100
n_abandonan       = int(df['abandono'].sum())
n_permanecen      = n_filas - n_abandonan
n_numericas       = df.drop(columns='abandono').select_dtypes(include='number').shape[1]
n_categoricas     = df.drop(columns='abandono').select_dtypes(include='object').shape[1]

# ── Métricas de configuración (leídas de config_modelado) ────────────────────
n_algoritmos_total = len(TODOS_LOS_ALGORITMOS)
cv_folds           = CV_CONFIG['cv_folds']
n_estrategias      = len(ESTRATEGIAS_BALANCE)

# ── Resumen en consola ────────────────────────────────────────────────────────
print('=' * 70)
print('DATASET: dataset_final_tfm.parquet')
print('=' * 70)
print(f'Filas:        {fmt(n_filas)}')
print(f'Columnas:     {n_cols} ({n_features} features + target)')
print(f'Abandono:     {fmt(n_abandonan)} ({pct_abandono:.1f}%)')
print(f'Permanecen:   {fmt(n_permanecen)} ({100 - pct_abandono:.1f}%)')
print(f'Numéricas:    {n_numericas} | Categóricas: {n_categoricas}')
print()
print(f'Algoritmos:   {n_algoritmos_total}')
print(f'CV:           {cv_folds}-Fold Stratified | random_state={CV_CONFIG["random_state"]}')
print(f'Estrategias:  {n_estrategias}')
print(f'Baseline:     {BASELINE_AUTOML["modelo"]} F1={BASELINE_AUTOML["f1_macro"]} | AUC={BASELINE_AUTOML["auc_roc"]}')

# ── Banner animado ────────────────────────────────────────────────────────────
mostrar_banner(
    titulo='Fase 5: Modelado Clásico',
    subtitulo=(
        f'{n_algoritmos_total} algoritmos · {cv_folds}-Fold CV · '
        f'Baseline F1={BASELINE_AUTOML["f1_macro"]}'
    ),
    icono='🤖',
    color='azul'
)

In [ ]:
# ============================================================================
# CELDA 3: GENERAR HTML
# ============================================================================
# Construye el contenido HTML y usa render_pagina_desde_fichero.
#
# render_pagina_desde_fichero('f5_m00_indice.ipynb', contenido, ...)
#   → extrae: título='Índice', fase='fase5', módulo='indice'
#   → llama render_base_html + generar_html_navegacion_completa internamente
#   → resultado: página completa con cabecera, nav, footer
#
# Es el ESTÁNDAR de todos los notebooks — no usar render_base_html directamente.
# ============================================================================

NOMBRE_FICHERO = 'f5_m00_indice.ipynb'  # Nombre real del notebook

# ── KPIs ─────────────────────────────────────────────────────────────────────
# 4 KPIs clave para contextualizar la fase
KPIS = [
    {'valor': fmt(n_filas),            'titulo': 'Expedientes',  'color': COLORES['primary']},
    {'valor': str(n_features),         'titulo': 'Features',     'color': COLORES['success']},
    {'valor': f'{pct_abandono:.1f}%',  'titulo': 'Abandono',     'color': COLORES['danger']},
    {'valor': str(n_algoritmos_total), 'titulo': 'Algoritmos',   'color': COLORES['warning']},
]

# ── Tarjetas de módulos (100% dinámicas desde MODULOS_FASE5) ──────────────────
# Las descripciones las genera config_modelado concatenando nombres de algoritmos.
# Si se añade un algoritmo allí, aquí se actualiza solo.
tarjetas = [
    {
        'titulo':      f'{m["id"].upper().replace("F5_", "")}: {m["nombre"]}',
        'descripcion': m['descripcion'],
        'emoji':       m['emoji'],
        'link':        m['archivo_html'],
        'link_texto':  'Ver módulo',
        'color':       m['color'],
    }
    for m in MODULOS_FASE5
]

# ── Contexto del proyecto ─────────────────────────────────────────────────────
metrica_ppal = next(m for m in METRICAS_EVALUAR if m['principal'])
contexto_html = (
    '<div style="background:#f0f4f8;padding:20px;border-radius:10px;'
    'border-left:4px solid #3182ce;">'
    f'<p><strong>Contexto:</strong> El screening AutoML (168 modelos, 4 frameworks) '
    f'identificó Gradient Boosting como familia óptima '
    f'({BASELINE_AUTOML["modelo"]}: F1={BASELINE_AUTOML["f1_macro"]}, '
    f'AUC={BASELINE_AUTOML["auc_roc"]}). '
    f'La Fase 4 EDA confirmó las features más relevantes.</p>'
    f'<p><strong>Objetivo Fase 5:</strong> Entrenamiento manual y controlado '
    f'de {n_algoritmos_total} algoritmos sobre {fmt(n_filas)} expedientes. '
    f'Métrica principal: <strong>{metrica_ppal["nombre"]}</strong>. '
    f'Selección del top-3 para Fase 6 (interpretabilidad + fairness).</p>'
    '</div>'
)

# ── Estrategia de validación ──────────────────────────────────────────────────
# Todos los valores vienen de CV_CONFIG y ESTRATEGIAS_BALANCE — cero hardcodes
nombres_estrategias = ' · '.join(e['nombre'] for e in ESTRATEGIAS_BALANCE)
nombres_metricas = ' · '.join(
    f'<strong>{m["nombre"]}</strong>' if m['principal'] else m['nombre']
    for m in METRICAS_EVALUAR
)
estrategia_html = (
    '<table style="width:100%;border-collapse:collapse;">'
    '<tr style="background:#3182ce;color:white;">'
    '<th style="padding:10px;text-align:left;width:200px">Aspecto</th>'
    '<th style="padding:10px;text-align:left;">Configuración</th></tr>'
    '<tr style="border-bottom:1px solid #e2e8f0;">'
    '<td style="padding:9px;"><strong>Split</strong></td>'
    f'<td style="padding:9px;">Train/Test estratificado '
    f'{int((1-CV_CONFIG["test_size"])*100)}/{int(CV_CONFIG["test_size"]*100)} '
    f'— random_state={CV_CONFIG["random_state"]}</td></tr>'
    '<tr style="border-bottom:1px solid #e2e8f0;">'
    '<td style="padding:9px;"><strong>Validación cruzada</strong></td>'
    f'<td style="padding:9px;">{cv_folds}-Fold Stratified CV sobre train</td></tr>'
    '<tr style="border-bottom:1px solid #e2e8f0;">'
    '<td style="padding:9px;"><strong>Balance</strong></td>'
    f'<td style="padding:9px;">{nombres_estrategias}</td></tr>'
    '<tr>'
    '<td style="padding:9px;"><strong>Métricas</strong></td>'
    f'<td style="padding:9px;">{nombres_metricas}</td></tr>'
    '</table>'
)

# ── Ensamblar contenido ───────────────────────────────────────────────────────
contenido = (
    generar_kpis_html(KPIS)
    + generar_seccion_html('Contexto del Proyecto', contexto_html)
    + generar_seccion_html('Módulos de la Fase', generar_tarjetas_html(tarjetas))
    + generar_seccion_html('Estrategia de Validación', estrategia_html)
)

# ── Renderizar con el ESTÁNDAR del proyecto ───────────────────────────────────
# render_pagina_desde_fichero extrae título/fase/módulo del nombre del fichero:
#   'f5_m00_indice.ipynb' → título='Índice', fase='fase5', módulo='indice'
render_pagina_desde_fichero(NOMBRE_FICHERO, contenido)

print(f'Tarjetas generadas: {len(tarjetas)}')

In [ ]:
# ============================================================================
# CELDA 4: RESUMEN FINAL
# ============================================================================

print('=' * 70)
print('F5-M00: ÍNDICE — RESUMEN')
print('=' * 70)
print(f'Dataset:      dataset_final_tfm.parquet ({fmt(n_filas)} filas, {n_features} features)')
print(f'Abandono:     {pct_abandono:.1f}%')
print(f'Algoritmos:   {n_algoritmos_total} en {len(MODULOS_FASE5)-1} módulos')
print(f'HTML:         docs/html/fase5/fase5_index.html')
print()
print('Siguiente: f5_m01a_preparacion.ipynb')

mostrar_resumen_final(
    titulo='F5-M00 completado',
    stats=[
        {'valor': fmt(n_filas),            'titulo': 'Expedientes',  'icono': '📋'},
        {'valor': str(n_features),         'titulo': 'Features',     'icono': '🔢'},
        {'valor': str(n_algoritmos_total), 'titulo': 'Algoritmos',   'icono': '🤖'},
        {'valor': str(len(tarjetas)),      'titulo': 'Módulos',      'icono': '📦'},
    ],
    icono='✅',
    color='azul'
)